# dataset_23: CICIoT2023 ML deneyi (Kişi 1)

Akış: Drive'ı bağla → repoyu çek → `src/model_ml.py`'yi çalıştır → sonuçlara bak → `outputs/dataset_23`'ü GitHub'a gönder.

**Bir kerelik hazırlık:**
1. `Merged01.csv` … `Merged05.csv` dosyalarını Drive'da `MyDrive/iot-data/dataset_23/` klasörüne yükle.
2. GitHub'da *Settings → Developer settings → Personal access tokens → **Tokens (classic)** → Generate new token* ile `repo` yetkili, süreli (ör. 90 gün) bir token oluştur. Repo başka bir hesapta olduğu için fine-grained token çalışmaz.
3. Colab'da sol menüdeki 🔑 **Secrets** bölümüne `GITHUB_TOKEN` adıyla ekle ve notebook erişimini aç. Token'ı asla hücreye yazma.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

GITHUB_USER = 'GamzeSerin'          # repo sahibi
REPO = 'bitirme-projesi'
BRANCH = 'dataset_23'                    # kendi dalın; main'e PR ile birleşir
DATA_DIR = '/content/drive/MyDrive/iot-data/dataset_23'
WORK_DIR = '/content/drive/MyDrive/iot-work/dataset_23'   # modeller + ara sonuçlar (Colab kopsa da kalır)

token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git'

In [ ]:
import os, subprocess
os.chdir('/content')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
os.chdir(f'/content/{REPO}')
!git fetch -q origin
!git checkout -q {BRANCH} 2>/dev/null || git checkout -q -b {BRANCH}
!git pull -q origin main --no-rebase
!pip install -q -r requirements.txt
!nvidia-smi -L || echo 'GPU yok: Runtime > Change runtime type > GPU'

## 1) Hızlı deneme (~5-10 dk)
Etiket başına 3.000 satır, 3 fold. Kodun uçtan uca çalıştığını görmek için. Çıktıyı ayrı bir klasöre yazar.

In [ ]:
!python src/model_ml.py --data-dir {DATA_DIR} --work-dir {WORK_DIR}_quick --out-dir outputs_quick/dataset_23 --quick --gpu

## 2) Asıl deney
Etiket başına en fazla 40.000 satır (~740 bin satır), 5-fold OOF, hiperparametre araması ve SHAP. `Number` özelliği atılır (ablation: skoru değiştirmiyor, ama veri toplama izi).
Uzun sürer. Colab koparsa hücreyi tekrar çalıştır: biten modeller `WORK_DIR`'den okunur, baştan eğitilmez.

In [ ]:
# Number veri toplama izi (floodlar 100, diğerleri 10 paketlik pencere) -> atılıyor
!python src/model_ml.py --data-dir {DATA_DIR} --work-dir {WORK_DIR} --tune --shap --gpu --drop-features Number

## 3) Sonuçlar: makale tabloları

In [ ]:
import json, pandas as pd
OUT = 'outputs/dataset_23'
metrics = json.load(open(f'{OUT}/metrics.json'))
manifest = json.load(open(f'{OUT}/manifest.json'))

summary = pd.DataFrame({m: {'OOF macro-F1': v['oof']['macro_f1'], 'Test macro-F1': v['test']['macro_f1'],
                            'Test accuracy': v['test']['accuracy'], 'Eğitim (s)': v['timing']['fit_seconds'],
                            'Tahmin (ms/1k)': v['timing']['predict_ms_per_1k']} for m, v in metrics.items()}).T
display(summary.sort_values('OOF macro-F1', ascending=False))

# Saldırı tipi x model (test F1): makalenin ana tablosu
f1 = pd.DataFrame({m: {c: v['test']['per_class'][c]['f1'] for c in manifest['classes']} for m, v in metrics.items()})
display(f1.style.highlight_max(axis=1).format('{:.4f}'))

# 34 ince etiket bazında recall: hangi alt saldırılar karışıyor?
fine = pd.DataFrame({m: v['test_fine_label_recall'] for m, v in metrics.items()})
display(fine.sort_values(manifest['primary_model']).head(15).style.format('{:.3f}'))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
m = manifest['primary_model']
cm = np.array(metrics[m]['test']['confusion_matrix'])
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cmn, display_labels=manifest['classes']).plot(ax=ax, values_format='.2f', xticks_rotation=45)
ax.set_title(f'{m}: normalize confusion matrix (test)')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP: sınıf başına en önemli 10 özellik
shap_top = json.load(open(f'{OUT}/shap_top_features.json'))
display(pd.DataFrame({c: [f['feature'] for f in v] for c, v in shap_top['per_class'].items()}))

## 4) DSS'i çalıştır ve sonuçları GitHub'a gönder

In [ ]:
!python main.py
!python main.py --demo dataset_23 --n 5

In [ ]:
!git config user.name 'ADIN SOYADIN'
!git config user.email 'github-mailin@example.com'
!git add outputs/dataset_23 src/model_ml.py
!git commit -m 'dataset_23: ML sonuçları güncellendi' || echo 'Değişiklik yok'
!git push -q origin {BRANCH}
print(f'GitHub\'da {BRANCH} → main için Pull Request aç.')